# Exploración guiada (punto de partida, Hito 1)

**Responsable:** D (con apoyo de todos)  
**Revisor:** (completar)

## Qué debe quedar resuelto

- [ ] Cargar `01-22-2020.csv`: primeras filas, número de filas y columnas, tipos de datos
- [ ] Contar los nulos por columna y explicar qué significan aquí
- [ ] Nombres a snake_case y fecha de actualización en AAAA-MM-DD
- [ ] Unir los reportes de enero a junio de 2020 (dos cambios de esquema)
- [ ] Top 10 de países al 30-jun-2020 y evolución de Chile (gráfico de líneas)
- [ ] Perfilado automático (ydata-profiling) y tres observaciones

In [8]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from src import config

## Cargar `01-22-2020.csv`: primeras filas, número de filas y columnas, tipos de datos

In [9]:
URL_BASE = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_daily_reports/"

df = pd.read_csv(URL_BASE + "01-22-2020.csv")

display(df.head())
print("Filas y columnas:", df.shape)
print(df.dtypes)

,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered
0,Anhui,Mainland China,1/22/2020 17:00,1.0,NaN,NaN
1,Beijing,Mainland China,1/22/2020 17:00,14.0,NaN,NaN
2,Chongqing,Mainland China,1/22/2020 17:00,6.0,NaN,NaN
3,Fujian,Mainland China,1/22/2020 17:00,1.0,NaN,NaN
4,Gansu,Mainland China,1/22/2020 17:00,NaN,NaN,NaN


Filas y columnas: (76, 6)
Province/State        str
Country/Region        str
Last Update           str
Confirmed         float64
Deaths            float64
Recovered         float64
dtype: object


## Contar los nulos por columna y explicar qué significan aquí

In [10]:
# Nulos por columna (cantidad y porcentaje)
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(1)

pd.DataFrame({"nulos": nulos, "porcentaje": porcentaje})

,nulos,porcentaje
Province/State,12,15.8
Country/Region,0,0.0
Last Update,0,0.0
Confirmed,10,13.2
Deaths,37,48.7
Recovered,37,48.7


In [11]:
# ¿Las filas sin Deaths son las mismas que sin Recovered?
print((df["Deaths"].isnull() == df["Recovered"].isnull()).all())

# Ver las filas sin Confirmed
df[df["Confirmed"].isnull()]

True


,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered
4,Gansu,Mainland China,1/22/2020 17:00,NaN,NaN,NaN
10,Heilongjiang,Mainland China,1/22/2020 17:00,NaN,NaN,NaN
12,Hong Kong,Hong Kong,1/22/2020 17:00,NaN,NaN,NaN
15,Inner Mongolia,Mainland China,1/22/2020 17:00,NaN,NaN,NaN
18,Jilin,Mainland China,1/22/2020 17:00,NaN,NaN,NaN
22,Qinghai,Mainland China,1/22/2020 17:00,NaN,NaN,NaN
23,Shaanxi,Mainland China,1/22/2020 17:00,NaN,NaN,NaN
30,Tibet,Mainland China,1/22/2020 17:00,NaN,NaN,NaN
32,Xinjiang,Mainland China,1/22/2020 17:00,NaN,NaN,NaN
38,Unknown,China,1/22/2020 17:00,NaN,NaN,NaN


In [12]:
df[df["Province/State"].isnull()]["Country/Region"].unique()

<StringArray>
[               'Japan',             'Thailand',          'South Korea',
             'Kiribati',                'Palau', 'Summer Olympics 2020',
                'Tonga', 'Winter Olympics 2022',           'Antarctica',
         'Korea, North',                'Nauru',               'Tuvalu']
Length: 12, dtype: str

In [13]:
# Filas con Province/State nulo y si traen algún dato
df[df["Province/State"].isnull()]

,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered
35,NaN,Japan,1/22/2020 17:00,2.0,NaN,NaN
36,NaN,Thailand,1/22/2020 17:00,4.0,0.0,2.0
37,NaN,South Korea,1/22/2020 17:00,1.0,NaN,NaN
39,NaN,Kiribati,1/22/2020 17:00,0.0,0.0,0.0
40,NaN,Palau,1/22/2020 17:00,0.0,0.0,0.0
42,NaN,Summer Olympics 2020,1/22/2020 17:00,0.0,0.0,0.0
60,NaN,Tonga,1/22/2020 17:00,0.0,0.0,0.0
61,NaN,Winter Olympics 2022,1/22/2020 17:00,0.0,0.0,0.0
62,NaN,Antarctica,1/22/2020 17:00,0.0,0.0,0.0
65,NaN,"Korea, North",1/22/2020 17:00,0.0,0.0,0.0


In [14]:
print((df["Confirmed"].notnull() & df["Deaths"].isnull()).sum())

27


In [15]:
print((df["Deaths"].isnull() == df["Recovered"].isnull()).all())

True


### Nulos por columna: qué significan en este conjunto

- **`Province/State` (12 nulos, 15,8 %):** hay dos casos distintos. Japan, Thailand y South Korea se reportaron a nivel nacional, sin provincia (ausencia de subdivisión). Las otras 9 filas (Kiribati, Palau, Tonga, Nauru, Tuvalu, Korea North, Antarctica, Summer Olympics 2020 y Winter Olympics 2022) son entidades sin casos que aparecen con ceros en las tres cifras. Son relleno del repositorio, no mediciones, y varias ni siquiera son países.
- **`Country/Region` y `Last Update` (0 nulos):** están completas. Son las llaves del reporte, así que es lo esperable.
- **`Confirmed` (10 nulos, 13,2 %):** corresponden a 10 filas de China (Gansu, Heilongjiang, Hong Kong, Inner Mongolia, Jilin, Qinghai, Shaanxi, Tibet, Xinjiang y "Unknown"), en las que `Confirmed`, `Deaths` y `Recovered` vienen vacías a la vez. Ese día esos territorios aún no informaban cifras. Un nulo no equivale a 0 casos: significa "no reportado".
- **`Deaths` y `Recovered` (37 nulos cada una, 48,7 %):** casi la mitad de las filas no informó fallecidos ni recuperados, y los nulos están en las mismas filas. De esas 37, 10 son los territorios de China sin ninguna cifra, y las 27 restantes informaron casos confirmados pero no fallecidos ni recuperados (por ejemplo, Japan y South Korea). Es el primer día del registro (22-ene-2020) y muchas jurisdicciones solo reportaban casos. Nulo significa "no informado", no 0.

**Observación de calidad:** en este reporte coexisten nulos ("no informado") y ceros de relleno. Un 0 no siempre es una medición real.

**Decisión:** no se imputan estos valores (imputar es inventar). Se mantienen como nulos y se documentan; más adelante se decidirá si se marcan o se excluyen según el análisis.

## Nombres a snake_case y fecha de actualización en AAAA-MM-DD

In [ ]:
# Normalizar nombres de columnas a snake_case
import re

def a_snake_case(nombre):
    nombre = nombre.strip()
    nombre = re.sub(r"[^0-9a-zA-Z]+", "_", nombre)   # "/", " ", "-" -> "_"
    return nombre.strip("_").lower()

df.columns = [a_snake_case(c) for c in df.columns]

# "1/22/2020 17:00" (mes/día/año hora) -> "2020-01-22"
df["last_update"] = pd.to_datetime(
    df["last_update"], format="%m/%d/%Y %H:%M"
).dt.strftime("%Y-%m-%d")

display(df.head())
print(df.columns.tolist())

,province_state,country_region,last_update,confirmed,deaths,recovered
0,Anhui,Mainland China,2020-01-22,1.0,NaN,NaN
1,Beijing,Mainland China,2020-01-22,14.0,NaN,NaN
2,Chongqing,Mainland China,2020-01-22,6.0,NaN,NaN
3,Fujian,Mainland China,2020-01-22,1.0,NaN,NaN
4,Gansu,Mainland China,2020-01-22,NaN,NaN,NaN


['province_state', 'country_region', 'last_update', 'confirmed', 'deaths', 'recovered']


In [17]:
# Verificamos que todo esté OK
import re

# 1. Todos los nombres de columna en snake_case
patron = re.compile(r"^[a-z0-9]+(_[a-z0-9]+)*$")
malos = [c for c in df.columns if not patron.match(c)]
print("Columnas fuera de snake_case:", malos)

# 2. Todas las fechas en AAAA-MM-DD
fechas_ok = df["last_update"].str.match(r"^\d{4}-\d{2}-\d{2}$")
print("Fechas con formato incorrecto:", (~fechas_ok).sum())

# 3. Que las fechas sean reales (no 2020-13-45)
fechas_validas = pd.to_datetime(df["last_update"], format="%Y-%m-%d", errors="coerce")
print("Fechas no válidas:", fechas_validas.isnull().sum())

# Si algo falla, el notebook se detiene
assert not malos, "Hay columnas fuera de snake_case"
assert fechas_ok.all(), "Hay fechas fuera de AAAA-MM-DD"
assert fechas_validas.notnull().all(), "Hay fechas inválidas"
print("Todo OK")

Columnas fuera de snake_case: []
Fechas con formato incorrecto: 0
Fechas no válidas: 0
Todo OK


## Unir los reportes de enero a junio de 2020 (dos cambios de esquema)

In [18]:
fechas = pd.date_range("2020-01-22", "2020-06-30")   # un reporte por día
renombres = {"lat": "latitude", "long": "longitude"}  # etapa 3 usa Lat/Long_

reportes = []
for fecha in fechas:
    nombre = fecha.strftime("%m-%d-%Y") + ".csv"
    try:
        d = pd.read_csv(URL_BASE + nombre)
    except Exception as e:
        print("No se pudo leer", nombre, "->", e)
        continue

    d.columns = [a_snake_case(c) for c in d.columns]
    d = d.rename(columns=renombres)
    d["fecha_reporte"] = fecha.strftime("%Y-%m-%d")   # fecha del archivo
    reportes.append(d)

# Las columnas que no existen en un reporte quedan como NaN
df_unido = pd.concat(reportes, ignore_index=True)

# Fecha de actualización a AAAA-MM-DD (aquí hay varios formatos distintos)
df_unido["last_update"] = pd.to_datetime(
    df_unido["last_update"], format="mixed"
).dt.strftime("%Y-%m-%d")

print("Reportes leídos:", len(reportes), "de", len(fechas))
print("Filas y columnas:", df_unido.shape)
print(df_unido.columns.tolist())

Reportes leídos: 161 de 161
Filas y columnas: (350945, 15)
['province_state', 'country_region', 'last_update', 'confirmed', 'deaths', 'recovered', 'fecha_reporte', 'latitude', 'longitude', 'fips', 'admin2', 'active', 'combined_key', 'incidence_rate', 'case_fatality_ratio']


In [ ]:
# Verificamos que todo esté OK
print((df_unido["last_update"] > df_unido["fecha_reporte"]).sum())   

242073


In [20]:
dif = (pd.to_datetime(df_unido["last_update"]) - pd.to_datetime(df_unido["fecha_reporte"])).dt.days

# 1. ¿De cuántos días es la diferencia?
print(dif.value_counts().sort_index())

# 2. ¿Desde qué fecha ocurre y en qué tipo de filas?
adelantadas = df_unido[dif > 0]
print(adelantadas["fecha_reporte"].min())
print(adelantadas["admin2"].notnull().mean())   # proporción con condado

-64     2
-63     2
-62     2
-61     2
-60     2
       ..
 327    2
 328    2
 329    2
 330    2
 331    2
Name: count, Length: 122, dtype: int64
2020-02-26
0.8548413082004189


In [21]:
dif = (pd.to_datetime(df_unido["last_update"]) - pd.to_datetime(df_unido["fecha_reporte"])).dt.days

# 1. Resumen compacto en vez de 122 filas
print("Mismo día o +1:", dif.between(0, 1).mean().round(3))
print("Negativas (stale):", (dif < 0).sum())
print("Más de 1 día adelante:", (dif > 1).sum())

# 2. Ejemplos de las adelantadas más extremas
cols = ["fecha_reporte", "last_update", "country_region", "province_state", "admin2"]
print(df_unido[dif > 30][cols].head(15))

Mismo día o +1: 0.989
Negativas (stale): 3843
Más de 1 día adelante: 112
       fecha_reporte last_update country_region province_state        admin2
153146    2020-05-06  2021-04-02             US         Alaska  Copper River
153147    2020-05-06  2021-04-02             US         Alaska       Chugach
156409    2020-05-07  2021-04-02             US         Alaska  Copper River
156410    2020-05-07  2021-04-02             US         Alaska       Chugach
159679    2020-05-08  2021-04-02             US         Alaska  Copper River
159680    2020-05-08  2021-04-02             US         Alaska       Chugach
162951    2020-05-09  2021-04-02             US         Alaska  Copper River
162952    2020-05-09  2021-04-02             US         Alaska       Chugach
166225    2020-05-10  2021-04-02             US         Alaska  Copper River
166226    2020-05-10  2021-04-02             US         Alaska       Chugach
169499    2020-05-11  2021-04-02             US         Alaska  Copper River
169

In [22]:
print(df_unido[dif > 1].groupby(["province_state", "admin2"]).size())

province_state  admin2      
Alaska          Chugach         56
                Copper River    56
dtype: int64


## Top 10 de países al 30-jun-2020 y evolución de Chile (gráfico de líneas)

In [ ]:
# TODO

## Perfilado automático (ydata-profiling) y tres observaciones

In [ ]:
# TODO